# Hourly BTC Realized GARCH Ensemble

This variant combines realized volatility measures, calibrated XGBoost probabilities, and a rolling circuit breaker. This walk-forward backtest predicts next-hour BTC/USDT price direction. It uses completed hourly candles from the public Binance spot API, including quote volume, trade counts, and taker-buy volume. Simulated payoffs are a fixed gain or loss of the position size based on whether the prediction is correct. Starting from the configured dates can take substantial time. Set END_DATE in the run cell to restrict the historical window. [Binance spot market data documentation](https://developers.binance.com/en/docs/catalog/core-trading-spot-trading/api/rest-api/market)



In [ ]:
from _binance_data import download_hourly_candles

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SelectFromModel
from sklearn.base import clone
from sklearn.metrics import log_loss
from arch import arch_model
import warnings
from collections import deque

warnings.filterwarnings('ignore')

# ==========================================
# SETTINGS - 1H INTERVAL
# ==========================================
SYMBOL = "BTCUSDT"

BASE_RISK = 0.01
MAX_POS = 0.05
VOL_GRID = [0.005, 0.010, 0.015, 0.020, 0.025, 0.030]
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ==========================================
# BINANCE DATA DOWNLOAD
# ==========================================

# ==========================================
# REALIZED MEASURES & GARCH - 1H
# ==========================================
def calculate_realized_measures(df_1h, window_hours=24):
    """Calculate realized variance, bipower variation, and jumps for the 24-hour GARCH input."""
    window = window_hours
    r = df_1h['log_ret']
    
    rv = (r ** 2).rolling(window).sum()
    
    mu_1 = np.sqrt(2 / np.pi)
    abs_r = r.abs()
    bpv = (abs_r * abs_r.shift(1)).rolling(window).sum() * (mu_1 ** -2)
    
    jump = np.maximum(rv - bpv, 0)
    rv_adj = rv - jump
    
    return rv, bpv, rv_adj, jump

def add_realized_garch_1h(df_1h):
    """EGARCH con Realized Measures - resample diario."""
    print("--> Calculating realized GARCH features...")
    
    start_idx = 168  # 1 semana
    df_clean = df_1h.iloc[start_idx:].replace([np.inf, -np.inf], np.nan).ffill()
    

    df_daily = df_clean.resample('1D').agg({
        'log_ret': 'sum',
        'RV_1h': 'last',
        'RV_6h': 'last',
        'RV_24h': 'last'
    }).dropna()
    
    if len(df_daily) < 100:
        df_1h['RGARCH_var'] = np.nan
        return df_1h
    

    rv_24h, _, rv_adj, _ = calculate_realized_measures(df_1h, window_hours=24)
    df_daily['RV_adj'] = df_clean.resample('1D')['log_ret'].apply(lambda x: (x**2).sum())
    
    window = 252
    forecasts = {}
    
    # Regressores HAR
    har_d = rv_adj.resample('1D').last().shift(1)
    har_w = rv_adj.rolling(168).mean().resample('1D').last().shift(1)
    regressors = pd.concat([har_d, har_w], axis=1).dropna().values * 10000
    regressors = np.clip(regressors, -np.nanpercentile(regressors, 99), np.nanpercentile(regressors, 99))
    
    for i in range(window, len(df_daily)):
        train_ret = df_daily['log_ret'].iloc[i-window:i] * 100
        x_train = regressors[i-window:i]
        
        try:
            model = arch_model(train_ret, x=x_train, mean='ARX', vol='EGARCH', 
                             p=1, o=1, q=1, dist='skewt')
            res = model.fit(disp='off', show_warning=False, 
                          options={'maxiter': 100, 'ftol': 1e-3})
            
            x_next = regressors[i].reshape(2, 1, 1)
            fcast = res.forecast(horizon=1, x=x_next, reindex=False)
            
            daily_var = fcast.variance.values[-1][0]
            daily_var = np.clip(daily_var, 0, 500)
            
            var_1h = (daily_var / 10000) / 24  # Dividir entre 24 horas
            forecasts[df_daily.index[i]] = var_1h
            
        except:
            if forecasts:
                forecasts[df_daily.index[i]] = list(forecasts.values())[-1]
            else:
                forecasts[df_daily.index[i]] = 0.0001
    
    s_forecast = pd.Series(forecasts)
    s_forecast_lagged = s_forecast.shift(1)
    
    df_1h['RGARCH_var'] = df_1h.index.normalize().map(s_forecast_lagged)
    df_1h['RGARCH_var'] = df_1h['RGARCH_var'].ffill(limit=72).shift(1)  # 72h buffer
    
    print(f"   Forecasts: {len(forecasts)}, Mean: {s_forecast.mean():.2e}")
    return df_1h

# ==========================================
# HAR FEATURES - 1H ADAPTED
# ==========================================
def add_har_features_1h(df_1h):
    """HAR Enhanced para 1H - ventanas: 1h, 6h, 24h, 168h (1 semana)."""
    print("--> Calculating enhanced hourly HAR features...")
    
    r = df_1h['log_ret']
    rv = r**2
    

    rv_1h = rv.shift(1)  # Lag 1
    rv_6h = rv.rolling(6).mean().shift(1)
    rv_24h = rv.rolling(24).mean().shift(1)
    rv_1w = rv.rolling(168).mean().shift(1)  # 168h = 1 semana
    
    # 1. Log-HAR
    df_1h['HAR_LogRV_1h'] = np.log(rv_1h + 1e-9)
    df_1h['HAR_LogRV_24h'] = np.log(rv_24h + 1e-9)
    df_1h['HAR_LogRV_1w'] = np.log(rv_1w + 1e-9)
    
    # 2. Continuous-Jump Decomposition
    mu_1 = np.sqrt(2/np.pi)
    abs_r = r.abs()
    
    bpv_24h = (abs_r * abs_r.shift(1)).rolling(24).sum() * (mu_1 ** -2)
    bpv_1w = (abs_r * abs_r.shift(1)).rolling(168).sum() * (mu_1 ** -2)
    
    df_1h['HAR_BPV_24h'] = bpv_24h.shift(1)
    df_1h['HAR_BPV_1w'] = bpv_1w.shift(1)
    df_1h['HAR_Jump_24h'] = np.maximum(rv.rolling(24).sum() - bpv_24h, 0).shift(1)
    df_1h['HAR_Jump_1w'] = np.maximum(rv.rolling(168).sum() - bpv_1w, 0).shift(1)
    
    # 3. Leverage Effect
    neg_rv = (r.clip(upper=0) ** 2)
    df_1h['HAR_Leverage_24h'] = neg_rv.rolling(24).sum().shift(1)
    df_1h['HAR_Leverage_1w'] = neg_rv.rolling(168).sum().shift(1)
    
    # 4. Signed Semivariances
    rs_pos = (r.clip(lower=0))**2
    rs_neg = neg_rv
    
    df_1h['HAR_RSpos_1h'] = rs_pos.shift(1)
    df_1h['HAR_RSpos_6h'] = rs_pos.rolling(6).sum().shift(1)
    df_1h['HAR_RSpos_24h'] = rs_pos.rolling(24).sum().shift(1)
    df_1h['HAR_RSpos_1w'] = rs_pos.rolling(168).sum().shift(1)
    df_1h['HAR_RSneg_1h'] = rs_neg.shift(1)
    df_1h['HAR_RSneg_6h'] = rs_neg.rolling(6).sum().shift(1)
    df_1h['HAR_RSneg_24h'] = rs_neg.rolling(24).sum().shift(1)
    df_1h['HAR_RSneg_1w'] = rs_neg.rolling(168).sum().shift(1)
    
    # 5. Volatility Regime
    vol_ma_1w = rv.rolling(168).mean()
    high_vol = (rv_24h > vol_ma_1w * 1.5).astype(int)
    df_1h['HAR_HighVol_Regime'] = high_vol.shift(1)
    df_1h['HAR_RV_24h_HighVol'] = (rv_24h * high_vol).shift(1)
    

    df_1h['HAR_RV_1h'] = rv.shift(1)
    df_1h['HAR_RV_6h'] = rv.rolling(6).mean().shift(1)
    df_1h['HAR_RV_24h'] = rv.rolling(24).mean().shift(1)
    df_1h['HAR_RV_1w'] = rv.rolling(168).mean().shift(1)
    
    # 7. Order Flow
    df_1h['taker_buy_ratio'] = df_1h['taker_base'] / (df_1h['Volume'] + 1e-9)
    df_1h['avg_trade_size'] = df_1h['quote_vol'] / (df_1h['trades'] + 1e-9)
    
    df_1h['Aggressor_Imbalance'] = ((df_1h['taker_base'] - (df_1h['Volume'] - df_1h['taker_base'])) / 
                                    (df_1h['Volume'] + 1e-9)).shift(1)
    df_1h['Quote_Pressure'] = (df_1h['taker_buy_ratio'] - 0.5).rolling(12).sum().shift(1)  # 12h
    df_1h['Large_Trade_Ratio'] = df_1h['avg_trade_size'].rolling(24).apply(
        lambda x: (x > x.quantile(0.9)).sum() / len(x) if len(x) > 0 else 0, raw=False
    ).shift(1)
    
    return df_1h

def calculate_rsi(series, periods=14):
    """RSI con suavizado Wilder."""
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/periods, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/periods, adjust=False).mean()
    rs = gain / (loss + 1e-9)
    rsi = 100 - 100/(1 + rs)
    return rsi.shift(1)

# ==========================================
# PREPARE FEATURES - 1H
# ==========================================
def prepare_features_1h(df):
    """Pipeline completo para 1H con HAR Enhanced + Order Flow."""
    print("--> Preparando features 1H...")
    print(f"[1] df inicial: {len(df)} rows")
    
    if len(df) < 2000:
        raise ValueError(f"Insufficient data: {len(df)} candles. A minimum of 2,000 candles is required.")
    
    df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna()
    print(f"[2] After log_ret: {len(df)} rows")


    df = add_har_features_1h(df)
    print(f"[3] After HAR Enhanced: {len(df)}")
    

    df['RV_1h'] = abs(df['log_ret']).shift(1)
    df['RV_6h'] = df['log_ret'].rolling(6).std().shift(1) * np.sqrt(6)
    df['RV_24h'] = df['log_ret'].rolling(24).std().shift(1) * np.sqrt(24)
    
    # GARCH
    df = add_realized_garch_1h(df)
    print(f"[4] After RGARCH: {len(df)}, NaN: {df['RGARCH_var'].isna().sum()}")

    # Microestructura
    df['RSI'] = calculate_rsi(df['Close'], periods=14)
    df['Vol_Shock'] = (df['Volume'].shift(1) / (df['Volume'].rolling(12).mean().shift(1) + 1e-9))
    df['Kurt_6h'] = df['log_ret'].rolling(6).kurt().shift(1)
    
    # Candlestick
    df['body_rel'] = ((df['Close'].shift(1) - df['Open'].shift(1)) / df['Open'].shift(1))
    df['wick_upper'] = ((df['High'].shift(1) - np.maximum(df['Open'].shift(1), df['Close'].shift(1))) / df['Open'].shift(1))
    df['wick_lower'] = ((np.minimum(df['Open'].shift(1), df['Close'].shift(1)) - df['Low'].shift(1)) / df['Open'].shift(1))
    

    df['num_trades'] = df['trades'].shift(1)
    df['hl_spread'] = ((df['High'] - df['Low']) / df['Close']).shift(1)
    
    # Garman-Klass
    log_hl = np.log(df['High'] / df['Low'])
    log_co = np.log(df['Close'] / df['Open'])
    df['GK_vol'] = (0.5 * log_hl**2 - (2*np.log(2)-1) * log_co**2).shift(1)
    
    # Risk metrics
    df['Downside_Dev'] = df['log_ret'].clip(upper=0).rolling(12).std().shift(1)
    
    # Efficiency
    change = (df['Close'] - df['Close'].shift(10)).abs()
    volatility = df['Close'].diff().abs().rolling(10).sum()
    df['KER'] = (change / (volatility + 1e-9)).shift(1)
    

    high_diff = df['High'].diff()
    low_diff = -df['Low'].diff()
    plus_dm = pd.Series(np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0), index=df.index)
    minus_dm = pd.Series(np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0), index=df.index)
    
    gk_norm = df['GK_vol'].rolling(14).mean() * df['Close']
    plus_di = 100 * plus_dm.rolling(14).mean() / (gk_norm + 1e-9)
    minus_di = 100 * minus_dm.rolling(14).mean() / (gk_norm + 1e-9)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    df['ADX'] = dx.rolling(14).mean().shift(1)
    
    # MA distance
    ma_20 = df['Close'].rolling(20).mean()
    df['Dist_MA'] = ((df['Close'] - ma_20) / ma_20).shift(1)
    
    # Vol of Vol
    df['Vol_of_Vol'] = df['RV_1h'].rolling(24).std().shift(1)
    
    # Tail risk (simplificado)
    df['Tail_Index'] = df['log_ret'].rolling(60).apply(
        lambda x: np.abs(x).nlargest(5).mean() if len(x) >= 5 else np.nan, raw=False
    ).shift(1)
    
    # Jump component
    mu_1 = np.sqrt(2/np.pi)
    abs_r = df['log_ret'].abs()
    bpv = (abs_r * abs_r.shift(1)).rolling(24).sum() * (mu_1 ** -2)
    jump_component = np.maximum(df['RV_1h'].rolling(24).sum() - bpv, 0)
    df['Jump_Burst'] = (jump_component / (df['RV_1h'].rolling(24).sum() + 1e-9)).shift(1)
    
    # Hurst (simplificado)
    df['Hurst'] = df['log_ret'].rolling(24).apply(
        lambda x: 0.5 + 0.1 * (x.std() / (x.abs().mean() + 1e-9)) if len(x) > 5 else 0.5, 
        raw=False
    ).shift(1)
    
    # Autocorrelation
    df['Corr_Shock'] = df['log_ret'].rolling(24).apply(
        lambda x: x.autocorr(lag=1) if len(x) > 1 else 0, raw=False
    ).shift(1)
    
    # Spread Z-score
    spread_raw = df['hl_spread']
    spread_ma = spread_raw.rolling(24).mean()
    spread_std = spread_raw.rolling(24).std()
    df['Spread_Z'] = ((spread_raw - spread_ma) / (spread_std + 1e-9)).shift(1)
    
    # Liquidity stress
    vol_norm = (df['Volume'] - df['Volume'].rolling(24).mean()) / (df['Volume'].rolling(24).std() + 1e-9)
    spread_norm = (df['hl_spread'] - df['hl_spread'].rolling(24).mean()) / (df['hl_spread'].rolling(24).std() + 1e-9)
    vol_stress = (df['RV_1h'] - df['RV_1h'].rolling(24).mean()) / (df['RV_1h'].rolling(24).std() + 1e-9)
    df['Liquidity_Stress'] = (-vol_norm + spread_norm + vol_stress).shift(1)
    
    # Momentum
    df['Mom_6h'] = df['Close'].pct_change(6).shift(1)
    df['Mom_24h'] = df['Close'].pct_change(24).shift(1)
    
    # Trend
    df['EMA_Fast'] = df['Close'].ewm(span=12).mean().shift(1)
    df['EMA_Slow'] = df['Close'].ewm(span=26).mean().shift(1)
    df['Trend'] = ((df['EMA_Fast'] - df['EMA_Slow']) / df['EMA_Slow'])
    
    # Missing value diagnostics
    print(f"[5] Missing value diagnostics:")
    for col in FEATURES_1H:
        if col in df.columns:
            nans = df[col].isna().sum()
            if nans > 0:
                print(f"   {col}: {nans} NaNs ({nans/len(df)*100:.1f}%)")
    
    # Dropna features
    total_antes = len(df)
    df = df.dropna(subset=FEATURES_1H)
    print(f"[6] After dropna: {len(df)} rows (removed {total_antes - len(df)})")
    
    # Target
    next_close = df['Close'].shift(-1)
    df['target'] = (next_close > df['Close']).astype(int)
    df = df.loc[next_close.notna()].copy()
    df['ret'] = df['Close'].pct_change()
    

    df = df.iloc[:-1]
    
    return df

# ==========================================
# FEATURES LISTA - 1H
# ==========================================
FEATURES_1H = [
    'RV_1h', 'RV_6h', 'RV_24h',
    'RSI', 'Vol_Shock', 'Kurt_6h',
    'body_rel', 'wick_upper', 'wick_lower',
    'RGARCH_var',
    
    # HAR Enhanced
    'HAR_LogRV_1h', 'HAR_LogRV_24h', 'HAR_LogRV_1w',
    'HAR_BPV_24h', 'HAR_BPV_1w',
    'HAR_Jump_24h', 'HAR_Jump_1w',
    'HAR_Leverage_24h', 'HAR_Leverage_1w',
    'HAR_RSpos_1h', 'HAR_RSpos_6h', 'HAR_RSpos_24h', 'HAR_RSpos_1w',
    'HAR_RSneg_1h', 'HAR_RSneg_6h', 'HAR_RSneg_24h', 'HAR_RSneg_1w',
    'HAR_HighVol_Regime', 'HAR_RV_24h_HighVol',
    'HAR_RV_1h', 'HAR_RV_6h', 'HAR_RV_24h', 'HAR_RV_1w',
    
    # Order Flow
    'Aggressor_Imbalance', 'Quote_Pressure', 'Large_Trade_Ratio',
    
    # Resto
    'taker_buy_ratio', 'avg_trade_size', 'num_trades', 'hl_spread',
    'GK_vol', 'Downside_Dev', 'KER', 'ADX', 'Dist_MA',
    'Vol_of_Vol', 'Tail_Index', 'Jump_Burst', 'Hurst', 'Corr_Shock',
    'Spread_Z', 'Liquidity_Stress', 'Mom_6h', 'Mom_24h', 'Trend'
]

# ==========================================
# ENSEMBLE OPTIMIZER (igual que 15m)
# ==========================================
def optimize_ensemble_weight(base_model, X_train, y_train):
    """Ensemble con clones independientes."""
    cv = TimeSeriesSplit(n_splits=3)
    
    oos_preds_sig = []
    oos_preds_iso = []
    oos_targets = []
    
    for train_idx, val_idx in cv.split(X_train):
        X_t, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_t, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        split = int(len(X_t) * 0.75)
        X_model, X_cal = X_t.iloc[:split], X_t.iloc[split:]
        y_model, y_cal = y_t.iloc[:split], y_t.iloc[split:]
        
        est_sig = clone(base_model)
        est_sig.fit(X_model, y_model)
        m_sig = CalibratedClassifierCV(FrozenEstimator(est_sig), method='sigmoid')
        m_sig.fit(X_cal, y_cal)
        
        est_iso = clone(base_model)
        est_iso.fit(X_model, y_model)
        m_iso = CalibratedClassifierCV(FrozenEstimator(est_iso), method='isotonic')
        m_iso.fit(X_cal, y_cal)
        
        oos_preds_sig.extend(m_sig.predict_proba(X_v)[:, 1])
        oos_preds_iso.extend(m_iso.predict_proba(X_v)[:, 1])
        oos_targets.extend(y_v)
    
    best_loss = 9999
    best_w = 0.5
    
    y_true = np.array(oos_targets)
    p_sig = np.array(oos_preds_sig)
    p_iso = np.array(oos_preds_iso)
    
    for w in np.linspace(0, 1, 21):
        p_blend = (w * p_sig) + ((1 - w) * p_iso)
        p_blend = np.clip(p_blend, 1e-15, 1 - 1e-15)
        loss = log_loss(y_true, p_blend)
        
        if loss < best_loss:
            best_loss = loss
            best_w = w
    
    split_final = int(len(X_train) * 0.75)
    X_model_f = X_train.iloc[:split_final]
    y_model_f = y_train.iloc[:split_final]
    X_cal_f = X_train.iloc[split_final:]
    y_cal_f = y_train.iloc[split_final:]
    
    est_sig_final = clone(base_model)
    est_sig_final.fit(X_model_f, y_model_f)
    final_sig = CalibratedClassifierCV(FrozenEstimator(est_sig_final), method='sigmoid')
    final_sig.fit(X_cal_f, y_cal_f)
    
    est_iso_final = clone(base_model)
    est_iso_final.fit(X_model_f, y_model_f)
    final_iso = CalibratedClassifierCV(FrozenEstimator(est_iso_final), method='isotonic')
    final_iso.fit(X_cal_f, y_cal_f)
    
    return best_w, final_sig, final_iso

# ==========================================
# IC CHECK
# ==========================================
def check_predictive_power(X, y):
    """IC test."""
    print("--> Validando IC...")
    
    X = X.replace([np.inf, -np.inf], np.nan).ffill().fillna(0)
    X = X.clip(-1e9, 1e9).astype(np.float32)

    cv = TimeSeriesSplit(n_splits=5)
    all_probs = []
    all_targets = []
    
    for train_idx, val_idx in cv.split(X):
        X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
        y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
        
        selector = xgb.XGBClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_SEED)
        selector.fit(X_t, y_t)
        selection = SelectFromModel(selector, prefit=True, threshold="0.25*mean")
        X_t_sel = X_t.iloc[:, selection.get_support()]
        X_v_sel = X_v.iloc[:, selection.get_support()]
        
        base = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.03,
                                 subsample=0.7, colsample_bytree=0.7,
                                 reg_alpha=0.1, reg_lambda=1.0,
                                 random_state=RANDOM_SEED, n_jobs=-1)
        cv_inner = TimeSeriesSplit(n_splits=3)
        model = CalibratedClassifierCV(base, method='sigmoid', cv=cv_inner)
        model.fit(X_t_sel, y_t)
        
        probs = model.predict_proba(X_v_sel)[:, 1]
        all_probs.extend(probs)
        all_targets.extend(y_v)
    
    ic = np.corrcoef(all_probs, all_targets)[0, 1]
    print(f"   IC: {ic:.4f}")
    
    return abs(ic) >= 0.02

# ==========================================
# BACKTESTER - 1H
# ==========================================
class Binance_Backtester_1h:
    def __init__(self, df, features):
        self.df = df
        self.features = features
        self.results = []
        self.BASE_RISK = BASE_RISK
        self.MAX_POS = MAX_POS
        self.VOL_GRID = VOL_GRID
        
        total_candles = len(df)
        if total_candles < 200:
            raise ValueError(f"Insufficient data: {total_candles} candles.")
        

        if total_candles >= 1500:
            self.TRAIN_SIZE = 1000
            self.TEST_SIZE = 180  # ~1 semana
        elif total_candles >= 750:
            self.TRAIN_SIZE = 500
            self.TEST_SIZE = 100
        else:
            self.TRAIN_SIZE = int(total_candles * 0.6)
            self.TEST_SIZE = int(total_candles * 0.15)
        
        print(f" Dataset: {total_candles} hourly candles")
        print(f"   Train Size: {self.TRAIN_SIZE} (~{self.TRAIN_SIZE/24:.1f} days)")
        print(f"   Test Size: {self.TEST_SIZE} (~{self.TEST_SIZE/24:.1f} days)")
    
    @staticmethod
    def clean_features(X):
        """Limpieza brutal para XGBoost."""
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.ffill().fillna(0)
        X = X.clip(-1e9, 1e9)
        return X.astype(np.float32)
    
    def select_best_features(self, X, y, min_features=15):
        X = self.clean_features(X)
        
        selector_model = xgb.XGBClassifier(
            n_estimators=50, max_depth=3, random_state=RANDOM_SEED, n_jobs=-1
        )
        selector_model.fit(X, y)
        
        selection = SelectFromModel(selector_model, prefit=True, threshold="0.25*mean")
        selected_features = X.columns[selection.get_support()].tolist()
        
        if len(selected_features) < min_features:
            imps = selector_model.feature_importances_
            indices = np.argsort(imps)[::-1][:min_features]
            selected_features = X.columns[indices].tolist()
        
        return selected_features
    
    def optimize_target_vol(self, X_train, y_train, vol_series, selected_features):
        split_point = int(len(X_train) * 0.75)
        
        X_model = self.clean_features(X_train[selected_features].iloc[:split_point])
        y_model = y_train.iloc[:split_point]
        
        X_val = self.clean_features(X_train[selected_features].iloc[split_point:])
        y_val = y_train.iloc[split_point:]
        vol_val = vol_series.iloc[split_point:]
        
        cv = TimeSeriesSplit(n_splits=3)
        model = xgb.XGBClassifier(
            n_estimators=100, max_depth=3, eval_metric='logloss',
            n_jobs=-1, random_state=RANDOM_SEED
        )
        calib = CalibratedClassifierCV(model, method='sigmoid', cv=cv)
        calib.fit(X_model, y_model)
        
        probs = calib.predict_proba(X_val)[:, 1]
        
        best_sharpe = -999
        best_vol = 0.015
        
        for v_cand in self.VOL_GRID:
            pnls = []
            for p, v, real in zip(probs, vol_val, y_val):
                signal = 0
                if p > 0.51: signal = 1
                elif p < 0.49: signal = -1
                if signal == 0:
                    pnls.append(0)
                    continue
                conviction = max(0, (p - 0.50) * 2)
                vol_scalar = np.clip(v_cand / max(v, 0.001), 0.5, 2.0)
                size = min(self.BASE_RISK * conviction * 10 * vol_scalar, self.MAX_POS)
                res = size if ((signal==1 and real==1) or (signal==-1 and real==0)) else -size
                pnls.append(res)
            
            if len(pnls) == 0:
                continue
            sharpe = np.mean(pnls) / (np.std(pnls) + 1e-9)
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_vol = v_cand
        
        return best_vol
        
    def optimize_thresholds(self, X_train_raw, y_train, vol_train, selected_features, best_vol):
        split_point = int(len(X_train_raw) * 0.75)
        
        X_model = self.clean_features(X_train_raw[selected_features].iloc[:split_point])
        y_model = y_train.iloc[:split_point]
        
        X_val = self.clean_features(X_train_raw[selected_features].iloc[split_point:])
        y_val = y_train.iloc[split_point:]
        vol_val = vol_train.iloc[split_point:]
        
        base = xgb.XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.7,
            n_jobs=-1, random_state=RANDOM_SEED
        )
        
        best_w, model_sig, model_iso = optimize_ensemble_weight(base, X_model, y_model)
        
        probs_sig = model_sig.predict_proba(X_val)[:, 1]
        probs_iso = model_iso.predict_proba(X_val)[:, 1]
        probs_val = (best_w * probs_sig) + ((1 - best_w) * probs_iso)
        
        best_sharpe = -999
        best_long = 0.507
        best_short = 0.493
        
        threshold_range = np.linspace(0.500, 0.530, 16)
        
        for long_thresh in threshold_range:
            short_thresh = 1.0 - long_thresh
            
            pnls = []
            for p, v, real in zip(probs_val, vol_val, y_val):
                signal = 0
                if p > long_thresh: 
                    signal = 1
                elif p < short_thresh: 
                    signal = -1
                
                if signal == 0:
                    pnls.append(0)
                    continue
                
                conviction = max(0, (abs(p - 0.50) - 0.01) * 2)
                vol_scalar = np.clip(best_vol / max(v, 0.001), 0.5, 2.0)
                size = min(self.BASE_RISK * conviction * 10 * vol_scalar, self.MAX_POS)
                
                res = size if ((signal==1 and real==1) or (signal==-1 and real==0)) else -size
                pnls.append(res)
            
            if len(pnls) == 0 or all(x == 0 for x in pnls):
                continue
            
            mean_pnl = np.mean(pnls)
            std_pnl = np.std(pnls)
            sharpe = mean_pnl / (std_pnl + 1e-9)
            
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_long = long_thresh
                best_short = short_thresh
        
        return best_long, best_short
    
    def run_walk_forward(self):
        """Run walk-forward validation with dynamic thresholds and a rolling circuit breaker."""
        print(f"--> Starting hourly walk-forward validation with dynamic thresholds and a circuit breaker...")
        

        X_first_train = self.df[self.features].iloc[0:self.TRAIN_SIZE]
        y_first_train = self.df['target'].iloc[0:self.TRAIN_SIZE]
        
        has_ic_power = check_predictive_power(X_first_train, y_first_train)
        if not has_ic_power:
            print(" IC bajo. El modelo puede no tener poder predictivo.")

        total_candles = len(self.df)
        total_steps = (total_candles - self.TRAIN_SIZE) // self.TEST_SIZE
        
        if total_steps < 1:
            total_steps = 1
        
        CONSTANT_B = 2.0
        KELLY_FRACTION = 0.25
        MIN_P_HIST = 0.48
        ROLLING_WINDOW = 50
        
        trade_history = deque(maxlen=ROLLING_WINDOW)
        
        print(f"   {total_steps} folds a ejecutar")
        print(f"   Kelly Config: b={CONSTANT_B}, Fraction={KELLY_FRACTION}")
        print(f"   Circuit Breaker: p_hist < {MIN_P_HIST*100:.0f}% (Rolling {ROLLING_WINDOW})")
        
        for i in range(total_steps):
            end_train = self.TRAIN_SIZE + i * self.TEST_SIZE
            start_train = 0
            
            start_test = end_train
            end_test = start_test + self.TEST_SIZE
            if end_test > total_candles:
                end_test = total_candles
            
            X_train_raw = self.df[self.features].iloc[start_train:end_train]
            y_train = self.df['target'].iloc[start_train:end_train]
            vol_train = self.df['RV_1h'].iloc[start_train:end_train]
            
            X_test_raw = self.df[self.features].iloc[start_test:end_test]
            y_test = self.df['target'].iloc[start_test:end_test]
            vol_test = self.df['RV_1h'].iloc[start_test:end_test]
            
            if len(X_test_raw) == 0:
                break
            
            best_feats = self.select_best_features(X_train_raw, y_train, min_features=15)
            
            X_train = self.clean_features(X_train_raw[best_feats])
            X_test = self.clean_features(X_test_raw[best_feats])
            
            best_vol = self.optimize_target_vol(X_train_raw, y_train, vol_train, best_feats)
            
            long_threshold, short_threshold = self.optimize_thresholds(
                X_train_raw, y_train, vol_train, best_feats, best_vol
            )
            
            base = xgb.XGBClassifier(
                n_estimators=200, learning_rate=0.03, max_depth=4,
                subsample=0.7, colsample_bytree=0.7,
                reg_alpha=0.1, reg_lambda=1.0,
                n_jobs=-1, random_state=RANDOM_SEED
            )
            
            best_w, model_sig, model_iso = optimize_ensemble_weight(base, X_train, y_train)
            
            probs_sig = model_sig.predict_proba(X_test)[:, 1]
            probs_iso = model_iso.predict_proba(X_test)[:, 1]
            probs_test = (best_w * probs_sig) + ((1 - best_w) * probs_iso)
            
            circuit_breaker_active_count = 0
            
            for j in range(len(X_test)):
                p_model = probs_test[j]
                v = vol_test.iloc[j]
                real = y_test.iloc[j]
                idx = X_test.index[j]
                
                p_hist = float(np.mean(trade_history)) if trade_history else 0.5
                
                raw_signal = 0
                if p_model > long_threshold: 
                    raw_signal = 1
                elif p_model < short_threshold: 
                    raw_signal = -1
                
                is_circuit_broken = p_hist < MIN_P_HIST
                
                size = 0
                kelly_factor = 0
                vol_scalar = 0
                
                if raw_signal != 0:
                    if not is_circuit_broken:
                        q_hist = 1 - p_hist
                        kelly_full = ((p_hist * CONSTANT_B) - q_hist) / CONSTANT_B
                        kelly_fractional = max(0, kelly_full * KELLY_FRACTION)
                        kelly_factor = kelly_fractional
                        
                        vol_scalar = np.clip(best_vol / max(v, 0.001), 0.5, 2.0)
                        
                        raw_size = self.BASE_RISK * 10 * vol_scalar * kelly_fractional
                        size = min(raw_size, self.MAX_POS)
                    else:
                        circuit_breaker_active_count += 1
                        size = 0

                pnl = 0
                trade_result = None
                
                if raw_signal != 0:
                    did_win = (raw_signal == 1 and real == 1) or (raw_signal == -1 and real == 0)
                    
                    if did_win:
                        trade_result = 1
                        pnl = size
                    else:
                        trade_result = 0
                        pnl = -size
                    
                    trade_history.append(trade_result)

                self.results.append({
                    'Date': idx,
                    'Prob_Model': p_model,
                    'P_Historical': p_hist,
                    'Kelly_Factor': kelly_factor if not is_circuit_broken else 0,
                    'Vol': v,
                    'Target_Vol': best_vol,
                    'Size': size,
                    'Signal': raw_signal,
                    'Outcome': real,
                    'PnL': pnl,
                    'Ensemble_W': best_w,
                    'Long_Threshold': long_threshold,
                    'Short_Threshold': short_threshold,
                    'Circuit_Breaker': 1 if is_circuit_broken else 0
                })
            
            wr_hist = float(np.mean(trade_history)) * 100 if trade_history else float('nan')
            cb_pct = (circuit_breaker_active_count / len(X_test)) * 100 if len(X_test) > 0 else 0
            
            print(f"   Step {i+1}/{total_steps}: Thresh=[{long_threshold:.3f}, {short_threshold:.3f}], "
                  f"WR_Roll={wr_hist:.1f}%, CB_Active={cb_pct:.1f}%")
        
        if len(self.results) == 0:
            print(" No se generaron resultados!")
            return pd.DataFrame()
        
        return pd.DataFrame(self.results).set_index('Date')

# ==========================================

# ==========================================


In [ ]:
START_DATE = "2021-01-01"
END_DATE = None  # Exclusive UTC endpoint. None uses the latest completed hour.

df_raw = download_hourly_candles(SYMBOL, START_DATE, END_DATE)
df = prepare_features_1h(df_raw)

try:
    backtester = Binance_Backtester_1h(df, FEATURES_1H)
    results = backtester.run_walk_forward()

    if len(results) > 0:
        results['Equity'] = (1 + results['PnL']).cumprod()
        results['BuyHold'] = (1 + df.loc[results.index, 'ret']).cumprod()

        trades = results[results['Size'] > 0]
        win_rate = len(trades[trades['PnL'] > 0]) / len(trades) * 100 if len(trades) > 0 else 0

        print("\n" + "="*50)
        print("DYNAMIC HOURLY BACKTEST RESULTS")
        print("="*50)
        print(f"Total Candles Evaluated:  {len(results)}")
        print(f"Simulated Trades: {len(trades)} ({len(trades)/len(results)*100:.1f}%)")
        print(f"Win Rate:               {win_rate:.2f}%")
        print(f"Strategy Return:     {(results['Equity'].iloc[-1]-1)*100:.2f}%")
        print(f"Buy-and-Hold Return:     {(results['BuyHold'].iloc[-1]-1)*100:.2f}%")
        print(f"Average Target Volatility:    {results['Target_Vol'].mean()*100:.2f}%")
        print(f"Average Position Size:          {trades['Size'].mean()*100:.2f}%" if len(trades) > 0 else "N/A")
        print(f"Average Ensemble Weight:    {results['Ensemble_W'].mean():.2f}")

        # Risk metrics
        equity = results['Equity']
        pnl = results['PnL']
        cum_max = equity.cummax()
        dd = equity / cum_max - 1.0
        max_dd = dd.min()

        candles_1h = len(results)
        years = candles_1h / (365.25 * 24)

        cagr = equity.iloc[-1] ** (1/years) - 1 if years > 0 else np.nan

        mean_pnl = pnl.mean()
        std_pnl = pnl.std()
        sharpe = (mean_pnl / (std_pnl + 1e-9)) * np.sqrt(365.25 * 24)

        ann_vol = std_pnl * np.sqrt(365.25 * 24)

        print("\n--- RISK METRICS ---")
        print(f"Strategy CAGR:        {cagr*100:8.2f}%")
        print(f"Max Drawdown:           {max_dd*100:8.2f}%")
        print(f"Annualized Volatility: {ann_vol*100:8.2f}%")
        print(f"Sharpe (anualizado):    {sharpe:8.2f}")

        # Circuit Breaker
        cb_triggered = results['Circuit_Breaker'].sum()
        cb_pct = cb_triggered / len(results) * 100

        print(f"\n--- CIRCUIT BREAKER ---")
        print(f"Candles blocked:       {cb_triggered} ({cb_pct:.1f}%)")
        print(f"Umbrales Long (avg):    {results['Long_Threshold'].mean():.3f}")
        print(f"Umbrales Short (avg):   {results['Short_Threshold'].mean():.3f}")

        # Charts
        fig, axes = plt.subplots(5, 1, figsize=(14, 14))

        # 1. Equity curve
        axes[0].plot(results.index, results['Equity'],
                     label='Strategy (Dynamic)', color='blue', lw=2)
        axes[0].plot(results.index, results['BuyHold'],
                     label='Buy & Hold', color='gray', alpha=0.5, ls='--')
        axes[0].axhline(1.0, color='black', lw=0.5)
        axes[0].set_yscale('log')
        axes[0].set_title(f"Equity Curve 1H | WinRate: {win_rate:.1f}%")
        axes[0].legend()
        axes[0].grid(alpha=0.3, which='both')

        # 2. Target Vol
        axes[1].plot(results.index, results['Target_Vol']*100,
                     color='green', label='Optimal Target Volatility')
        axes[1].set_title("Target Volatility")
        axes[1].set_ylabel("Vol %")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        # 3. Position Sizing
        if len(trades) > 0:
            sc = axes[2].scatter(trades.index, trades['Size']*100,
                                 c=trades['PnL'], cmap='RdYlGn',
                                 alpha=0.6, s=20)
            axes[2].set_title("Position Sizing (color = PnL)")
            axes[2].set_ylabel("Size %")
            axes[2].grid(alpha=0.3)

        # 4. Ensemble Weight
        axes[3].plot(results.index, results['Ensemble_W'], color='purple', alpha=0.7)
        axes[3].axhline(0.5, color='gray', ls='--', alpha=0.3)
        axes[3].set_ylim(0, 1)
        axes[3].set_ylabel('Weight Sigmoid')
        axes[3].set_title('Ensemble Weight Evolution')
        axes[3].grid(alpha=0.3)

        # 5. Dynamic Thresholds
        axes[4].plot(results.index, results['Long_Threshold'], 
                    color='green', label='Long Threshold', alpha=0.7)
        axes[4].plot(results.index, results['Short_Threshold'], 
                    color='red', label='Short Threshold', alpha=0.7)
        axes[4].axhline(0.5, color='gray', ls='--', alpha=0.3)
        axes[4].set_ylabel('Threshold')
        axes[4].set_title('Dynamic Thresholds Evolution')
        axes[4].legend()
        axes[4].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print(" No trades were generated by the backtest.")

except Exception:
    raise
